# IPv6 Probe — Validate §08 Hetzner Replacement Hypothesis

Goal: figure out if Colab can replace the planned €3.79/mo Hetzner VPS for IPv6-based YouTube rotation.

Tests 4 hypotheses sequentially:
- **H1** Does Colab runtime have IPv6 egress?
- **H2** Does Colab have a /64 subnet (multiple bindable addrs)?
- **H3** Does YouTube accept IPv6 (HTML + InnerTube)?
- **H4** Does YouTube treat distinct /128 as distinct clients (no /64 lump-limit)?

Decision matrix at the bottom (Cell 8).

In [ ]:
# ===== Cell 1: Setup =====
!pip install --quiet requests 2>/dev/null

import socket, subprocess, time, json, sys, os, random
from contextlib import contextmanager
import urllib.request
import ipaddress

RESULTS = {}

def run(cmd, timeout=10):
    try:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
        return r.stdout.strip(), r.stderr.strip(), r.returncode
    except subprocess.TimeoutExpired:
        return '', 'TIMEOUT', -1

print('Python:', sys.version.split()[0])
print('Platform:', sys.platform)
out, _, _ = run('uname -a')
print('Kernel:', out)
print('---')
print('Setup OK')

In [ ]:
# ===== Cell 2: H1 — IPv6 capability probe =====
print('=== H1: IPv6 capability ===\n')

# (a) interface list
out, _, _ = run('ip -6 addr show')
print('--- ip -6 addr show ---')
print(out or '(empty)')
print()

# (b) default route v6
out, _, _ = run('ip -6 route show default')
print('--- ip -6 route show default ---')
print(out or '(no v6 default route)')
print()

# (c) outbound IPv6
ip6, err6, rc6 = run('curl -6 -s -m 8 https://ifconfig.co')
print('--- curl -6 ifconfig.co ---')
print(f'rc={rc6}  out={ip6!r}  err={err6!r}')
print()

# (d) outbound IPv4 (control)
ip4, err4, rc4 = run('curl -4 -s -m 8 https://ifconfig.co')
print('--- curl -4 ifconfig.co ---')
print(f'rc={rc4}  out={ip4!r}  err={err4!r}')
print()

# (e) DNS AAAA resolution
try:
    info = socket.getaddrinfo('www.youtube.com', 443, socket.AF_INET6)
    aaaa = sorted({ai[4][0] for ai in info})
    print(f'--- youtube AAAA --- ({len(aaaa)} unique)')
    for a in aaaa[:5]: print(' ', a)
    h1_dns_ok = True
except Exception as e:
    print(f'--- youtube AAAA --- FAIL: {e}')
    aaaa = []
    h1_dns_ok = False

h1_egress = (rc6 == 0 and ':' in ip6)
RESULTS['H1'] = {
    'egress_ok': h1_egress,
    'ipv6_egress': ip6 if h1_egress else None,
    'ipv4_egress': ip4 if rc4 == 0 else None,
    'dns_aaaa_ok': h1_dns_ok,
    'youtube_aaaa_count': len(aaaa),
}
print(f"\n>>> H1 verdict: {'PASS' if h1_egress else 'FAIL'} (egress={h1_egress}, dns_aaaa={h1_dns_ok})")

In [ ]:
# ===== Cell 3: H2 — /64 subnet probe (can we bind multiple IPv6 source addrs?) =====
print('=== H2: /64 subnet probe ===\n')

if not RESULTS.get('H1', {}).get('egress_ok'):
    print('SKIP — H1 failed, no IPv6')
    RESULTS['H2'] = {'skipped': True}
else:
    # parse our IPv6 address
    out, _, _ = run('ip -6 addr show scope global')
    my_v6 = None
    for line in out.splitlines():
        line = line.strip()
        if line.startswith('inet6 '):
            cand = line.split()[1].split('/')[0]
            if not cand.startswith('fe80'):
                my_v6 = cand; break
    print(f'Our global IPv6: {my_v6}')
    
    if not my_v6:
        print('No global IPv6 addr on interfaces (despite egress working — strange)')
        RESULTS['H2'] = {'has_global': False, 'multi_bind_ok': False}
    else:
        # try binding 5 different /128 addrs in the assumed /64
        net = ipaddress.IPv6Network(f'{my_v6}/64', strict=False)
        candidates = [str(net.network_address + i) for i in range(2, 12)]
        bind_results = []
        for addr in candidates[:5]:
            try:
                s = socket.socket(socket.AF_INET6, socket.SOCK_STREAM)
                s.bind((addr, 0))
                # try connecting outbound
                s.settimeout(5)
                try:
                    s.connect(('2001:4860:4860::8888', 443))  # google IPv6 DNS over TLS port
                    conn_ok = True
                except Exception as e:
                    conn_ok = False; conn_err = str(e)
                s.close()
                bind_results.append({'addr': addr, 'bind': True, 'connect': conn_ok})
                print(f'  {addr}  bind=OK  connect={"OK" if conn_ok else "FAIL"}')
            except Exception as e:
                bind_results.append({'addr': addr, 'bind': False, 'err': str(e)})
                print(f'  {addr}  bind=FAIL  {e}')
        
        bind_ok = sum(1 for r in bind_results if r.get('connect'))
        RESULTS['H2'] = {
            'has_global': True,
            'our_v6': my_v6,
            'subnet': str(net),
            'tested': len(bind_results),
            'connect_ok': bind_ok,
            'multi_bind_ok': bind_ok >= 3,
            'detail': bind_results,
        }
        print(f"\n>>> H2 verdict: {'PASS' if bind_ok >= 3 else 'FAIL'} ({bind_ok}/{len(bind_results)} bindable+routable)")

In [ ]:
# ===== Cell 4: H3a — YouTube HTML direct via IPv6 (latency vs IPv4) =====
print('=== H3a: YouTube HTML IPv6 ===\n')

if not RESULTS.get('H1', {}).get('egress_ok'):
    print('SKIP — no IPv6')
    RESULTS['H3a'] = {'skipped': True}
else:
    import requests, urllib3
    urllib3.disable_warnings()

    UA = 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36'
    URL = 'https://www.youtube.com/@MrBeast/videos'

    def fetch(family):
        # monkey-patch getaddrinfo to force family
        orig = socket.getaddrinfo
        def patched(host, port, fam=0, *a, **kw):
            return orig(host, port, family, *a, **kw)
        socket.getaddrinfo = patched
        try:
            t = time.perf_counter()
            r = requests.get(URL, headers={'User-Agent': UA}, timeout=15)
            dt = time.perf_counter() - t
            return {'status': r.status_code, 'len': len(r.content), 'sec': round(dt, 3)}
        except Exception as e:
            return {'error': str(e)}
        finally:
            socket.getaddrinfo = orig

    v6_runs = [fetch(socket.AF_INET6) for _ in range(3)]
    v4_runs = [fetch(socket.AF_INET) for _ in range(3)]

    print('IPv6 runs:', v6_runs)
    print('IPv4 runs:', v4_runs)

    v6_ok = sum(1 for r in v6_runs if r.get('status') == 200)
    v4_ok = sum(1 for r in v4_runs if r.get('status') == 200)
    avg6 = sum(r['sec'] for r in v6_runs if 'sec' in r) / max(1, sum(1 for r in v6_runs if 'sec' in r))
    avg4 = sum(r['sec'] for r in v4_runs if 'sec' in r) / max(1, sum(1 for r in v4_runs if 'sec' in r))

    RESULTS['H3a'] = {
        'v6_success': f'{v6_ok}/3',
        'v4_success': f'{v4_ok}/3',
        'v6_avg_sec': round(avg6, 3),
        'v4_avg_sec': round(avg4, 3),
        'ratio_v6_over_v4': round(avg6 / avg4, 2) if avg4 > 0 else None,
    }
    print(f"\n>>> H3a verdict: v6 {v6_ok}/3 ({avg6:.2f}s), v4 {v4_ok}/3 ({avg4:.2f}s)")

In [ ]:
# ===== Cell 5: H3b — InnerTube /browse via IPv6 =====
print('=== H3b: InnerTube /browse IPv6 ===\n')

if not RESULTS.get('H1', {}).get('egress_ok'):
    print('SKIP — no IPv6')
    RESULTS['H3b'] = {'skipped': True}
else:
    import requests

    # 5 channel IDs to test (mix of large channels)
    BROWSE_IDS = [
        'UCX6OQ3DkcsbYNE6H8uQQuVA',  # MrBeast
        'UC-lHJZR3Gqxm24_Vd_AJ5Yw',  # PewDiePie
        'UCq-Fj5jknLsUf-MWSy4_brA',  # T-Series
        'UCbCmjCuTUZos6Inko4u57UQ',  # Cocomelon
        'UCFFbwnve3yF62-tVXkTyHqg',  # 5-Minute Crafts
    ]
    INNERTUBE_URL = 'https://www.youtube.com/youtubei/v1/browse?prettyPrint=false'
    BODY_TMPL = {
        'context': {
            'client': {
                'clientName': 'WEB',
                'clientVersion': '2.20240101.00.00',
                'hl': 'en', 'gl': 'US',
            }
        },
    }

    def innertube(family, cid):
        orig = socket.getaddrinfo
        socket.getaddrinfo = lambda h, p, *a, **kw: orig(h, p, family)
        try:
            body = dict(BODY_TMPL); body['browseId'] = cid
            t = time.perf_counter()
            r = requests.post(INNERTUBE_URL, json=body,
                              headers={'User-Agent': 'Mozilla/5.0', 'Content-Type': 'application/json'},
                              timeout=15)
            dt = time.perf_counter() - t
            return {'status': r.status_code, 'len': len(r.content), 'sec': round(dt, 3)}
        except Exception as e:
            return {'error': str(e)[:80]}
        finally:
            socket.getaddrinfo = orig

    v6 = [innertube(socket.AF_INET6, c) for c in BROWSE_IDS]
    v4 = [innertube(socket.AF_INET, c) for c in BROWSE_IDS]
    for i, (a, b) in enumerate(zip(v6, v4)):
        print(f'  ch{i}  v6={a}  v4={b}')

    v6_ok = sum(1 for r in v6 if r.get('status') == 200)
    v4_ok = sum(1 for r in v4 if r.get('status') == 200)
    RESULTS['H3b'] = {
        'v6_success': f'{v6_ok}/5',
        'v4_success': f'{v4_ok}/5',
        'v6_avg_sec': round(sum(r['sec'] for r in v6 if 'sec' in r) / max(1, v6_ok), 3) if v6_ok else None,
        'v4_avg_sec': round(sum(r['sec'] for r in v4 if 'sec' in r) / max(1, v4_ok), 3) if v4_ok else None,
    }
    print(f"\n>>> H3b verdict: v6 {v6_ok}/5, v4 {v4_ok}/5")

In [ ]:
# ===== Cell 6: H4 — multi-IPv6 source rotation (does YT see them as distinct?) =====
print('=== H4: multi-IPv6 rotation ===\n')

if not RESULTS.get('H2', {}).get('multi_bind_ok'):
    print('SKIP — H2 failed, no rotation possible')
    RESULTS['H4'] = {'skipped': True}
else:
    import requests
    my_v6 = RESULTS['H2']['our_v6']
    net = ipaddress.IPv6Network(f'{my_v6}/64', strict=False)
    # 30 random /128 in the /64
    candidates = [str(net.network_address + random.randint(100, 2**60)) for _ in range(30)]

    INNERTUBE_URL = 'https://www.youtube.com/youtubei/v1/browse?prettyPrint=false'
    BODY = {
        'context': {'client': {'clientName': 'WEB', 'clientVersion': '2.20240101.00.00'}},
        'browseId': 'UCX6OQ3DkcsbYNE6H8uQQuVA',
    }

    def call_with_source(src):
        # bind a source via a custom adapter
        from urllib3.poolmanager import PoolManager
        from requests.adapters import HTTPAdapter
        class SourceAdapter(HTTPAdapter):
            def init_poolmanager(self, *a, **k):
                k['socket_options'] = []
                k['source_address'] = (src, 0)
                self.poolmanager = PoolManager(*a, **k)
        s = requests.Session()
        s.mount('https://', SourceAdapter())
        orig = socket.getaddrinfo
        socket.getaddrinfo = lambda h, p, *a, **kw: orig(h, p, socket.AF_INET6)
        try:
            t = time.perf_counter()
            r = s.post(INNERTUBE_URL, json=BODY,
                       headers={'User-Agent': 'Mozilla/5.0', 'Content-Type': 'application/json'},
                       timeout=12)
            return {'src': src, 'status': r.status_code, 'sec': round(time.perf_counter()-t, 3)}
        except Exception as e:
            return {'src': src, 'error': str(e)[:70]}
        finally:
            socket.getaddrinfo = orig

    out = []
    for i, src in enumerate(candidates):
        r = call_with_source(src)
        out.append(r)
        if i < 5 or 'error' in r or r.get('status') != 200:
            print(f'  [{i}] {r}')

    succ = sum(1 for r in out if r.get('status') == 200)
    rate = succ / len(out)
    RESULTS['H4'] = {
        'attempts': len(out),
        'success': succ,
        'success_rate': round(rate, 3),
        'all_sources_worked': rate >= 0.95,
    }
    print(f"\n>>> H4 verdict: {succ}/{len(out)} = {rate*100:.1f}%   {'PASS' if rate>=0.95 else 'PARTIAL/FAIL'}")

In [ ]:
# ===== Cell 7: Throughput test (best case) =====
print('=== Cell 7: throughput ===\n')

if not RESULTS.get('H4', {}).get('all_sources_worked'):
    print('SKIP — multi-source rotation not viable, no point measuring throughput')
    RESULTS['THROUGHPUT'] = {'skipped': True}
else:
    import requests, concurrent.futures as cf
    my_v6 = RESULTS['H2']['our_v6']
    net = ipaddress.IPv6Network(f'{my_v6}/64', strict=False)
    sources = [str(net.network_address + random.randint(100, 2**60)) for _ in range(50)]
    BODY = {
        'context': {'client': {'clientName': 'WEB', 'clientVersion': '2.20240101.00.00'}},
        'browseId': 'UCX6OQ3DkcsbYNE6H8uQQuVA',
    }

    def one(src):
        from urllib3.poolmanager import PoolManager
        from requests.adapters import HTTPAdapter
        class SA(HTTPAdapter):
            def init_poolmanager(self, *a, **k):
                k['source_address'] = (src, 0)
                self.poolmanager = PoolManager(*a, **k)
        s = requests.Session(); s.mount('https://', SA())
        orig = socket.getaddrinfo
        socket.getaddrinfo = lambda h,p,*a,**k: orig(h,p,socket.AF_INET6)
        try:
            r = s.post('https://www.youtube.com/youtubei/v1/browse?prettyPrint=false',
                       json=BODY, headers={'User-Agent':'Mozilla/5.0','Content-Type':'application/json'},
                       timeout=15)
            return r.status_code
        except Exception as e:
            return str(e)[:50]
        finally:
            socket.getaddrinfo = orig

    t = time.perf_counter()
    with cf.ThreadPoolExecutor(max_workers=10) as ex:
        results = list(ex.map(one, sources))
    elapsed = time.perf_counter() - t
    ok = sum(1 for r in results if r == 200)
    rate = ok / elapsed
    print(f'50 InnerTube calls / {elapsed:.1f}s = {rate:.2f} req/s   ({ok}/50 success)')
    RESULTS['THROUGHPUT'] = {
        'elapsed_sec': round(elapsed, 2),
        'success': ok,
        'attempts': len(sources),
        'req_per_sec': round(rate, 2),
        'vs_ipv4_spike_715': round(rate / 7.15, 2),
    }

In [ ]:
# ===== Cell 8: Verdict =====
import json
print('=== FINAL RESULTS ===\n')
print(json.dumps(RESULTS, indent=2, default=str))
print()

H1 = RESULTS.get('H1', {}).get('egress_ok', False)
H2 = RESULTS.get('H2', {}).get('multi_bind_ok', False)
H3 = (str(RESULTS.get('H3a', {}).get('v6_success', '0/0')).split('/')[0] not in ('0','None','skipped') and
      str(RESULTS.get('H3b', {}).get('v6_success', '0/0')).split('/')[0] not in ('0','None','skipped'))
H4 = RESULTS.get('H4', {}).get('all_sources_worked', False)

print('Hypothesis results:')
print(f'  H1 (IPv6 egress):           {"✓" if H1 else "✗"}')
print(f'  H2 (/64 multi-bind):        {"✓" if H2 else "✗"}')
print(f'  H3 (YouTube accepts v6):    {"✓" if H3 else "✗"}')
print(f'  H4 (per-/128 distinct):     {"✓" if H4 else "✗"}')
print()

if H1 and H2 and H3 and H4:
    print('>>> VERDICT: PERFECT — replace Hetzner with Colab IPv6 runtime.')
elif H1 and H2 and H3 and not H4:
    print('>>> VERDICT: PARTIAL — YT treats /64 as one client. Use cookie pool instead of IPv6 rotation.')
elif H1 and not H2 and H3:
    print('>>> VERDICT: SINGLE IPv6 only — no rotation value. Equivalent to single-IP spike.')
elif H1 and not H3:
    print('>>> VERDICT: YouTube refuses IPv6 — abandon v6 path.')
else:
    print('>>> VERDICT: NO IPv6 on Colab — fall back to Hetzner VPS (§08 original).')